# Moshi Family Test

포크 `latentforge/transformers-moshi` (5.15.0.dev0) 기준.

`MoshiProcessor`는 feature extractor / tokenizer / **Mimi 코덱**을 한 번에 묶습니다.
일반 오디오 모델과 달리 `input_values`를 내지 않고, 스트림별로 **이미 인코딩된 코드**를 냅니다:

- `audio=` → `user_audio_codes` (사용자 발화)
- `assistant_audio=` → `assistant_audio_codes` (Moshi 자신의 발화, 프롬프트용)
- `text=` → `input_ids` (오디오 프레임 수에 맞춰 자동 pad)

In [1]:
import transformers

print(transformers.__version__)
print([n for n in dir(transformers) if "Moshi" in n or "Mimi" in n])

/data/Haan/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5.15.0.dev0
['MimiConfig', 'MimiModel', 'MimiPreTrainedModel', 'MoshiConfig', 'MoshiDepthConfig', 'MoshiDepthDecoderForCausalLM', 'MoshiDepthDecoderModel', 'MoshiForCausalLM', 'MoshiForConditionalGeneration', 'MoshiModel', 'MoshiPreTrainedModel', 'MoshiProcessor']


## 1. 프로세서 조립

`MoshiProcessor(feature_extractor, tokenizer, audio_tokenizer, num_codebooks=8)`.
`audio_tokenizer`는 `MimiModel` 인스턴스이고, `num_codebooks`는 코덱이 아니라 **Moshi 체크포인트**의 속성입니다
(Mimi는 Moshi가 읽는 것보다 많은 코드북을 낼 수 있음).

In [2]:
import torch
from transformers import (
    AutoFeatureExtractor,
    AutoTokenizer,
    MimiModel,
    MoshiForConditionalGeneration,
    MoshiProcessor,
)

MODEL_ID = "kmhf/hf-moshiko"
MIMI_ID = "kyutai/mimi"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

mimi = MimiModel.from_pretrained(MIMI_ID).to(device).eval()
processor = MoshiProcessor(
    feature_extractor=AutoFeatureExtractor.from_pretrained(MIMI_ID),
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID),
    audio_tokenizer=mimi,
    num_codebooks=8,
)

model = MoshiForConditionalGeneration.from_pretrained(MODEL_ID, dtype=dtype).to(device).eval()

# 저장해두면 다음부터 MoshiProcessor.from_pretrained(...) 한 줄로 로드된다.
# processor.save_pretrained("./moshi-processor")

sr = mimi.config.sampling_rate
frame_rate = mimi.config.frame_rate
print(f"device={device} dtype={dtype} sr={sr} frame_rate={frame_rate}")


Loading weights:   0%|          | 0/350 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 350/350 [00:00<00:00, 5765.62it/s]


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 53430.62it/s]


Loading weights:   0%|          | 0/333 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 333/333 [00:00<00:00, 4970.28it/s]

device=cuda dtype=torch.bfloat16 sr=24000 frame_rate=12.5


## 2. 입력 인코딩

텍스트는 오디오 프레임 수에 맞춰 자동으로 오른쪽 pad 됩니다
(텍스트가 더 길면 `ValueError` — 오디오를 늘리거나 텍스트를 줄여야 함).

In [3]:
import numpy as np

# 실제 파일: librosa.load("user.wav", sr=sr, mono=True)[0]
seconds = 5.0
user_audio = np.zeros(int(sr * seconds), dtype=np.float32)

inputs = processor(text="Hello", audio=user_audio, sampling_rate=sr)

for k, v in inputs.items():
    print(f"{k:24s} {tuple(v.shape)}")

input_ids                (1, 63)
attention_mask           (1, 63)
user_audio_codes         (1, 8, 63)
assistant_audio_codes    (1, 8, 63)


## 3. 생성

프로세서 출력 키(`input_ids`, `user_audio_codes`, `assistant_audio_codes`)가
`generate`의 인자명과 그대로 맞아서 `**inputs`로 바로 넘어갑니다.

In [4]:
torch.manual_seed(0)  # 샘플링 재현성

MAX_NEW_TOKENS = 64

inputs = inputs.to(device)

# Moshi는 full-duplex라 매 스텝 사용자 프레임을 하나씩 소비한다. 프롬프트만 주면 생성 구간이
# 비어서 generate가 거부하므로, 생성할 프레임 수만큼 무음을 이어 붙인다.
# (모델이 사용자 스트림까지 예측하는 체크포인트라면 이 연장이 없어도 된다.)
silence = processor.get_silence_audio_codes(MAX_NEW_TOKENS + 1, batch_size=1).to(device)
inputs["user_audio_codes"] = torch.cat([inputs["user_audio_codes"], silence], dim=2)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=0.8,
        top_k=250,
        return_audio_codes=True,
    )

text_out = processor.tokenizer.decode(output.sequences[0], skip_special_tokens=True)
print("text:", text_out)
print("audio_codes:", tuple(output.audio_codes.shape))


[transformers] Passing `generation_config` together with generation-related arguments=({'output_scores', 'return_dict_in_generate'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=64) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Using `cache_implementation='sliding_window' is deprecated and will be removed in v5.13. Please only use one of ('static', 'offloaded_static'), and the layer structure will be inferred automatically.


W0720 06:33:21.345000 3586838 torch/_dynamo/variables/tensor.py:1759] [0/0] Graph break from `Tensor.item()`, consider setting:
W0720 06:33:21.345000 3586838 torch/_dynamo/variables/tensor.py:1759] [0/0]     torch._dynamo.config.capture_scalar_outputs = True
W0720 06:33:21.345000 3586838 torch/_dynamo/variables/tensor.py:1759] [0/0] or:
W0720 06:33:21.345000 3586838 torch/_dynamo/variables/tensor.py:1759] [0/0]     env TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS=1
W0720 06:33:21.345000 3586838 torch/_dynamo/variables/tensor.py:1759] [0/0] to include these operations in the captured graph.
W0720 06:33:21.345000 3586838 torch/_dynamo/variables/tensor.py:1759] [0/0] 
W0720 06:33:21.345000 3586838 torch/_dynamo/variables/tensor.py:1759] [0/0] Graph break: from user code at:
W0720 06:33:21.345000 3586838 torch/_dynamo/variables/tensor.py:1759] [0/0]   File "/data/transformers-moshi/src/transformers/utils/generic.py", line 911, in wrapper
W0720 06:33:21.345000 3586838 torch/_dynamo/variables/tensor.p

W0720 06:33:24.466000 3586838 torch/_dynamo/convert_frame.py:1994] [9/8] torch._dynamo hit config.recompile_limit (8)
W0720 06:33:24.466000 3586838 torch/_dynamo/convert_frame.py:1994] [9/8]    function: '__call__' (/data/transformers-moshi/src/transformers/modeling_layers.py:76)
W0720 06:33:24.466000 3586838 torch/_dynamo/convert_frame.py:1994] [9/8]    last reason: 9/7: self._modules['self_attn'].layer_idx == 1                # keys, values = self.layers[layer_idx].update(key_states, value_states, *args, **kwargs)  # transformers-moshi/src/transformers/cache_utils.py:1304 in update (HINT: torch.compile considers integer attributes of the nn.Module to be static. If you are observing recompilation, you might want to make this integer dynamic using torch._dynamo.config.allow_unspec_int_on_nn_module = True, or convert this integer into a tensor.)
W0720 06:33:24.466000 3586838 torch/_dynamo/convert_frame.py:1994] [9/8] User stack trace:
W0720 06:33:24.466000 3586838 torch/_dynamo/convert_

W0720 06:33:36.263000 3586838 torch/_inductor/cudagraph_utils.py:516] [__cudagraphs] CUDAGraph supports dynamic shapes by recording a new graph for each distinct input size. Recording too many CUDAGraphs may lead to extra overhead. We have observed 9 distinct sizes. Please consider the following options for better performance: a) padding inputs to a few fixed number of shapes; or b) set torch._inductor.config.triton.cudagraph_skip_dynamic_graphs=True. Set torch._inductor.config.triton.cudagraph_dynamic_shape_warn_limit=None to silence this warning.


text: Hello Hello, what's going on?
audio_codes: (1, 8, 126)


In [5]:
from IPython.display import Audio

# 코드 -> 파형은 프로세서가 담당 (내부적으로 Mimi decode)
with torch.no_grad():
    waveform = processor.decode_audio(output.audio_codes)
print(tuple(waveform.shape), waveform.shape[-1] / sr, "sec")

Audio(waveform[0, 0].float().cpu().numpy(), rate=sr)


(1, 1, 241920) 10.08 sec


## 4. 라이브 사용자 없이 생성 — `get_silence_audio_codes`

Moshi는 full-duplex라 매 스텝 user 프레임을 소비합니다. 사용자가 없으면 무음을 넣어야 하는데,
Mimi는 스트리밍 코덱이라 **무음이 상수 코드로 양자화되지 않습니다** (컨볼루션 상태가 프레임을 넘어 이어짐).
한 프레임을 반복하면 안 되고, 구간 전체를 한 번에 인코딩해야 해서 이 헬퍼가 있습니다.

In [6]:
torch.manual_seed(0)  # 샘플링 재현성

NEW_TOKENS = 32

# 사용자 스트림의 길이가 곧 프롬프트 길이가 된다. 라이브 사용자가 없는 생성이라면 프롬프트는
# 한 프레임이면 충분하고, 나머지는 생성 구간을 덮을 무음이다.
# Moshi는 매 스텝 사용자 프레임을 하나씩 소비하므로 지평선 전체를 덮어야 한다.
silence_codes = processor.get_silence_audio_codes(num_frames=1 + NEW_TOKENS + 1, batch_size=1).to(device)
print("silence:", tuple(silence_codes.shape))  # (B, num_codebooks, num_frames)

# 한 프레임 반복 != 실제 무음 인코딩 (Mimi는 스트리밍 코덱이라 프레임마다 코드가 다르다)
print("frame 0 == frame 10?", torch.equal(silence_codes[..., 0], silence_codes[..., 10]))

# 프롬프트는 한 프레임. 텍스트/assistant 스트림은 "아직 말한 적 없음"으로 채워진다.
uncond = model.get_unconditional_inputs(num_samples=1)
with torch.no_grad():
    out = model.generate(
        input_ids=uncond.input_ids.to(device),
        assistant_audio_codes=uncond.assistant_audio_codes.to(device),
        user_audio_codes=silence_codes,
        max_new_tokens=NEW_TOKENS,
        do_sample=True,
    )
print(processor.tokenizer.decode(out.sequences[0], skip_special_tokens=True))


[transformers] Both `max_new_tokens` (=32) and `max_length`(=34) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


silence: (1, 8, 34)
frame 0 == frame 10? False


2033,


## 5. 양쪽 스트림 프롬프트

`assistant_audio`로 "Moshi가 이미 말한 내용"을 함께 주면 대화 중간부터 이어갈 수 있습니다.

In [7]:
assistant_audio = np.zeros(int(sr * seconds), dtype=np.float32)  # 실제로는 Moshi의 이전 발화

duplex = processor(
    text="Hi there",
    audio=user_audio,
    assistant_audio=assistant_audio,
    sampling_rate=sr,
).to(device)

print(sorted(duplex.keys()))
print({k: tuple(v.shape) for k, v in duplex.items()})

['assistant_audio_codes', 'attention_mask', 'input_ids', 'user_audio_codes']
{'input_ids': (1, 63), 'attention_mask': (1, 63), 'user_audio_codes': (1, 8, 63), 'assistant_audio_codes': (1, 8, 63)}


## 6. 스트리밍 대화 (chunk-by-chunk full-duplex)

**주의:** Moshi의 `generate`에는 내장 스트리밍/스텝 API가 없습니다 (`streamer` 인자 없음).
실시간 대화는 아래 세 가지를 직접 이어붙여 구현합니다:

| 상태 | 무엇 | 왜 필요한가 |
|---|---|---|
| `padding_cache` | `MimiConv1dPaddingCache` | Mimi causal conv의 프레임 간 패딩 상태. 없으면 청크 경계마다 클릭 노이즈 |
| `encoder_past_key_values` | Mimi encoder transformer KV | 코덱 인코더의 시간축 문맥 |
| `past_key_values` | Moshi temporal transformer KV | 대화 문맥. 이게 없으면 매 청크가 새 대화 |

그리고 2번째 청크부터는 **`concat_unconditional_inputs=False`** — 안 그러면 매 청크마다
unconditional BOS 프레임이 다시 붙습니다.

`use_streaming=True`로 부르면 Mimi가 `padding_cache`를 알아서 만들어 반환하므로,
첫 청크는 `None`으로 넘기고 이후엔 받은 걸 그대로 되먹이면 됩니다.

In [8]:
class MoshiStreamingSession:
    """청크 단위 full-duplex 세션. 사용자 오디오를 밀어넣으면 Moshi 오디오가 나온다."""

    def __init__(self, model, processor, num_codebooks=8):
        self.model = model
        self.processor = processor
        self.mimi = processor.audio_tokenizer
        self.num_codebooks = num_codebooks
        # "오디오 없음" placeholder 코드. Mimi 코드북은 0..audio_vocab_size-1 뿐이다.
        self.audio_pad = model.config.audio_vocab_size
        self.reset()

    def reset(self):
        self.padding_cache = None          # Mimi conv 패딩 상태
        self.encoder_pkv = None            # Mimi encoder KV
        self.decoder_pkv = None            # Mimi decoder KV
        self.past_key_values = None        # Moshi KV (= 대화 문맥)
        self.first = True
        self.emitted = 0            # 이미 파형으로 내보낸 프레임 수

        # generate()는 매번 세 스트림을 다 요구하고, 셋의 길이가 서로 맞아야 한다.
        # 따라서 직전 1프레임이 아니라 대화 전체를 누적해서 들고 있어야 한다.
        uncond = self.model.get_unconditional_inputs(num_samples=1)
        self.hist_ids = uncond.input_ids                        # (1, T)
        self.hist_assistant = uncond.assistant_audio_codes      # (1, K, T)
        # 첫 청크는 concat_unconditional_inputs가 프레임을 하나 더 붙이므로 2프레임 확보한다.
        self.hist_user = self.processor.get_silence_audio_codes(num_frames=2, batch_size=1)

    @torch.no_grad()
    def push(self, chunk: np.ndarray, **gen_kwargs):
        """chunk: (samples,) float32 @ 24kHz. 반환: (waveform, text)"""
        wav = torch.as_tensor(chunk, dtype=torch.float32, device=self.mimi.device)
        wav = wav.reshape(1, 1, -1)
        n_prev = self.hist_ids.shape[-1]

        # 1) 사용자 오디오 -> 코드 (스트리밍 인코딩, conv/KV 상태 되먹임)
        enc = self.mimi.encode(
            wav,
            num_quantizers=self.num_codebooks,
            encoder_past_key_values=self.encoder_pkv,
            padding_cache=self.padding_cache,
            use_streaming=True,
            return_dict=True,
        )
        self.encoder_pkv = enc.encoder_past_key_values
        self.padding_cache = enc.padding_cache

        # 사용자 스트림도 누적한다. 이 프레임들이 이번 청크에서 모델이 소비할 입력이다.
        self.hist_user = torch.cat([self.hist_user, enc.audio_codes], dim=-1)
        n_new = enc.audio_codes.shape[-1]

        # 2) Moshi 한 청크 생성
        out = self.model.generate(
            input_ids=self.hist_ids,
            assistant_audio_codes=self.hist_assistant,
            user_audio_codes=self.hist_user,
            max_new_tokens=n_new,                  # 프레임 수만큼만 (full-duplex 정렬)
            concat_unconditional_inputs=self.first,
            return_audio_codes=True,
            **gen_kwargs,
        )
        self.first = False

        # 히스토리 갱신. assistant 쪽은 out.audio_codes(= delay를 되돌린 출력)가 아니라
        # 모델이 들고 있는 비-delay 히스토리를 받아야 다음 호출에서 의미가 맞는다.
        self.hist_ids = out.sequences
        self.hist_assistant = self.model.generated_audio_codes.clone()

        # 이번 청크에서 새로 생성된 텍스트만 (안 자르면 청크마다 전체가 중복된다)
        text = self.processor.tokenizer.decode(out.sequences[0, n_prev:], skip_special_tokens=True)

        # 3) Moshi 코드 -> 파형.
        # out.audio_codes 는 이번 청크분이 아니라 대화 전체를 담는다. 그리고 delay pattern 때문에
        # 뒤쪽 코드북이 아직 도착하지 않은 프레임에는 자리표시자(= audio_vocab_size)가 남아 있는데,
        # 그 값은 Mimi 코드북 범위 밖이라 그대로 decode 하면 device-side assert 로 죽는다.
        # 그래서 (a) 모든 코드북이 유효한 프레임만 남기고 (b) 이미 내보낸 만큼은 건너뛴다.
        # 잘린 프레임은 사라지지 않는다 — delay 가 채워지는 다음 호출에서 완성되어 나온다.
        codes = out.audio_codes
        if codes is None or codes.shape[-1] == 0:
            return np.zeros(0, dtype=np.float32), text
        complete = (codes < self.audio_pad).all(dim=1).all(dim=0)
        codes = codes[..., complete]
        new_frames = codes[..., self.emitted :]
        self.emitted = codes.shape[-1]
        if new_frames.shape[-1] == 0:
            return np.zeros(0, dtype=np.float32), text

        # decode() 는 encode() 와 달리 use_streaming/padding_cache 를 받지 않아 디코더 conv
        # 상태는 청크 간 이어지지 않는다 (경계 아티팩트가 남을 수 있다).
        dec = self.mimi.decode(
            new_frames,
            decoder_past_key_values=self.decoder_pkv,
            return_dict=True,
        )
        self.decoder_pkv = dec.decoder_past_key_values
        return dec.audio_values[0, 0].float().cpu().numpy(), text


### 실행 — 80ms 청크로 밀어넣기

Mimi 프레임레이트가 12.5Hz면 1프레임 = 80ms. 청크를 프레임 경계에 맞춰야
인코더가 부분 프레임을 버리지 않습니다.

In [9]:
torch.manual_seed(0)  # 샘플링 재현성

samples_per_frame = int(sr / frame_rate)      # 24000 / 12.5 = 1920
frames_per_chunk = 5                          # 5프레임 = 400ms
chunk_samples = samples_per_frame * frames_per_chunk
print(f"{samples_per_frame} samples/frame, chunk = {chunk_samples} samples")

session = MoshiStreamingSession(model, processor)

# 실제로는 마이크 스트림. 여기서는 5초 무음을 청크로 쪼갠다.
stream = np.zeros(int(sr * 5.0), dtype=np.float32)
n_chunks = len(stream) // chunk_samples

audio_out, text_out = [], []
for i in range(n_chunks):
    chunk = stream[i * chunk_samples : (i + 1) * chunk_samples]
    wav, text = session.push(chunk, do_sample=True, temperature=0.8, top_k=250)
    audio_out.append(wav)
    if text.strip():
        text_out.append(text)
    print(f"chunk {i:2d}/{n_chunks}  out={wav.shape[0]:5d} samples  text={text!r}")

full = np.concatenate(audio_out)
print("\ntotal:", full.shape, full.shape[0] / sr, "sec")
print("transcript:", " ".join(text_out))

[transformers] Both `max_new_tokens` (=5) and `max_length`(=7) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1920 samples/frame, chunk = 9600 samples


[transformers] Both `max_new_tokens` (=5) and `max_length`(=12) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


chunk  0/12  out= 7680 samples  text='as it was'


[transformers] Both `max_new_tokens` (=5) and `max_length`(=17) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=5) and `max_length`(=57) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


chunk  9/12  out= 7680 samples  text=''


[transformers] Both `max_new_tokens` (=5) and `max_length`(=62) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


chunk 10/12  out= 7680 samples  text=''


chunk 11/12  out= 7680 samples  text=''

total: (94080,) 3.92 sec
transcript: as it was still a book of a year, you know what I mean.


In [10]:
Audio(full, rate=sr)

### 스트리밍이 제대로 붙었는지 확인

청크로 나눠 인코딩한 결과가 통짜 인코딩과 (거의) 같아야 합니다.
`padding_cache`를 되먹이지 않으면 여기서 어긋납니다.

In [11]:
probe = np.random.RandomState(0).randn(chunk_samples * 4).astype(np.float32) * 0.1
probe_t = torch.as_tensor(probe, device=mimi.device).reshape(1, 1, -1)

with torch.no_grad():
    whole = mimi.encode(probe_t, num_quantizers=8).audio_codes

    pc, pkv, parts = None, None, []
    for i in range(4):
        seg = probe_t[..., i * chunk_samples : (i + 1) * chunk_samples]
        e = mimi.encode(
            seg, num_quantizers=8, encoder_past_key_values=pkv,
            padding_cache=pc, use_streaming=True, return_dict=True,
        )
        pkv, pc = e.encoder_past_key_values, e.padding_cache
        parts.append(e.audio_codes)
    chunked = torch.cat(parts, dim=-1)

print("whole  :", tuple(whole.shape))
print("chunked:", tuple(chunked.shape))
n = min(whole.shape[-1], chunked.shape[-1])
agree = (whole[..., :n] == chunked[..., :n]).float().mean().item()
print(f"code agreement: {agree:.1%}")

whole  : (1, 8, 20)
chunked: (1, 8, 20)
code agreement: 100.0%


## 7. 학습 경로 (KD용)

`forward`는 `text_labels` / `audio_labels`를 받습니다. Depth Transformer 단독 클래스도 노출돼 있어
텍스트 스트림과 오디오 스트림 로짓을 분리해 distill할 수 있습니다.

In [12]:
import inspect

from transformers import MoshiDepthDecoderForCausalLM  # noqa: F401

print(inspect.signature(MoshiForConditionalGeneration.forward))

# 학습 경로는 세 스트림이 정확히 같은 길이여야 한다 (generate 와 달리 사용자 스트림을 미리
# 더 받지 않는다). 위에서 만든 duplex 입력이 그 조건을 만족한다.
with torch.no_grad():
    out = model(
        **duplex,
        text_labels=duplex["input_ids"],
        audio_labels=duplex["assistant_audio_codes"],
    )
print("loss:", out.loss)


(self, input_ids: torch.LongTensor | None = None, attention_mask: torch.BoolTensor | None = None, user_audio_codes: torch.Tensor | None = None, assistant_audio_codes: torch.Tensor | None = None, past_key_values: transformers.cache_utils.Cache | None = None, inputs_embeds: torch.FloatTensor | None = None, text_labels: torch.LongTensor | None = None, audio_labels: torch.LongTensor | None = None, use_cache: bool | None = None, output_attentions: bool | None = None, output_hidden_states: bool | None = None, **kwargs) -> transformers.models.moshi.modeling_moshi.MoshiConditionalGenerationOutputWithPast


loss: tensor(13.1226, device='cuda:0')
